# CG4002: Voice PyTorch 1D-CNN Trainer & HLS Export

This notebook mirrors the gesture training workflow, but for voice classification.


## 1. Import dependencies & configuration

In [ ]:
# Install required packages (uncomment if needed)
# %pip install numpy pandas scikit-learn matplotlib seaborn

In [ ]:
# Install PyTorch + Torchaudio (uncomment if needed)
# %pip install torch torchaudio

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

# CONFIGURATION
SAMPLE_RATE = 16000
N_MFCC = 40
TARGET_FRAMES = 50
NUM_CLASSES = 3
BATCH_SIZE = 16
EPOCHS = 40
LEARNING_RATE = 0.001
AUGMENT_FACTOR = 3
RANDOM_SEED = 42

AUDIO_ROOT = Path('../data/audio')
MANIFEST_CSV = Path('../data/voice_manifest.csv')
FEATURES_NPY = Path('../data/voice_features.npy')
LABELS_NPY = Path('../data/voice_labels.npy')
TRAIN_NPY = Path('../data/voice_X_train.npy')
TEST_NPY = Path('../data/voice_X_test.npy')
YTRAIN_NPY = Path('../data/voice_y_train.npy')
YTEST_NPY = Path('../data/voice_y_test.npy')
MEAN_NPY = Path('../data/voice_mean.npy')
STD_NPY = Path('../data/voice_std.npy')

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 2. Data processing & augmentation

### 2.1 Scan and label raw voice files

In [ ]:
# Expected folder layout:
# ../data/audio/
#   class0/
#     *.wav
#   class1/
#     *.wav
#   class2/
#     *.wav

def scan_audio_dataset(root: Path):
    if not root.exists():
        raise FileNotFoundError(f'Audio folder not found: {root.resolve()}')

    class_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if len(class_dirs) == 0:
        raise RuntimeError(f'No class subfolders found in: {root.resolve()}')

    class_map = {d.name: i for i, d in enumerate(class_dirs)}

    rows = []
    for class_dir in class_dirs:
        label_id = class_map[class_dir.name]
        for wav in sorted(class_dir.glob('*.wav')):
            rows.append({
                'path': str(wav),
                'label': class_dir.name,
                'label_id': label_id,
            })

    if len(rows) == 0:
        raise RuntimeError('No .wav files found under class folders.')

    manifest = pd.DataFrame(rows)
    return manifest, class_map

manifest_df, CLASS_MAP = scan_audio_dataset(AUDIO_ROOT)
manifest_df.to_csv(MANIFEST_CSV, index=False)

print(f'✅ Found {len(manifest_df)} voice samples')
print('Class map:', CLASS_MAP)
print(manifest_df.head())


### 2.2 Convert waveform to fixed MFCC windows

In [ ]:
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={
        'n_fft': 512,
        'win_length': 400,
        'hop_length': 160,
        'n_mels': 40,
        'center': True,
        'power': 2.0,
    }
)

def fix_mfcc_time(mfcc_2d, target_frames=TARGET_FRAMES):
    # mfcc_2d: [40, T]
    t = mfcc_2d.shape[1]
    if t == target_frames:
        return mfcc_2d
    if t > target_frames:
        return mfcc_2d[:, :target_frames]
    pad = target_frames - t
    return torch.nn.functional.pad(mfcc_2d, (0, pad), mode='constant', value=0.0)


def load_wav_to_mfcc(path):
    wav, sr = torchaudio.load(path)  # [ch, n]
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

    feat = mfcc_transform(wav).squeeze(0)  # [40, T]
    feat = fix_mfcc_time(feat, TARGET_FRAMES)

    # Per-coefficient normalization across time
    mu = feat.mean(dim=1, keepdim=True)
    sd = feat.std(dim=1, keepdim=True).clamp_min(1e-6)
    feat = (feat - mu) / sd

    return feat.numpy().astype(np.float32)


### 2.3 Audio augmentation

In [ ]:
def augment_waveform(wav, noise_std=0.003, gain_low=0.85, gain_high=1.15):
    # wav shape: [1, N]
    aug = wav.clone()

    # Random gain
    gain = torch.empty(1).uniform_(gain_low, gain_high).item()
    aug = aug * gain

    # Additive gaussian noise
    noise = torch.randn_like(aug) * noise_std
    aug = aug + noise

    return aug.clamp(-1.0, 1.0)

all_X = []
all_y = []

for _, row in manifest_df.iterrows():
    wav_path = row['path']
    label_id = int(row['label_id'])

    wav, sr = torchaudio.load(wav_path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

    # original + augmented variants
    versions = [wav]
    for _ in range(AUGMENT_FACTOR - 1):
        versions.append(augment_waveform(wav))

    for v in versions:
        feat = mfcc_transform(v).squeeze(0)
        feat = fix_mfcc_time(feat, TARGET_FRAMES)
        mu = feat.mean(dim=1, keepdim=True)
        sd = feat.std(dim=1, keepdim=True).clamp_min(1e-6)
        feat = (feat - mu) / sd

        all_X.append(feat.numpy().astype(np.float32))   # [40, 50]
        all_y.append(label_id)

X = np.stack(all_X, axis=0)  # [N, 40, 50]
y = np.array(all_y, dtype=np.int64)

np.save(FEATURES_NPY, X)
np.save(LABELS_NPY, y)

print('✅ Features generated')
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Class counts:', {int(i): int((y == i).sum()) for i in np.unique(y)})


## 3. Model definition & training

### 3.1 Load Data

In [ ]:
X = np.load(FEATURES_NPY).astype(np.float32)
y = np.load(LABELS_NPY).astype(np.int64)

# Split with stratification
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

# Dataset-level normalization (fit on train only)
train_mean = X_train.mean(axis=(0, 2), keepdims=True)  # [1,40,1]
train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

# Save normalization (flatten to 40 for deployment convenience)
np.save(MEAN_NPY, train_mean.reshape(-1).astype(np.float32))
np.save(STD_NPY, train_std.reshape(-1).astype(np.float32))

np.save(TRAIN_NPY, X_train)
np.save(TEST_NPY, X_test)
np.save(YTRAIN_NPY, y_train)
np.save(YTEST_NPY, y_test)

print(f'Raw shapes: X_train={X_train.shape}, X_test={X_test.shape}')
print('Saved train/test split and voice_mean.npy / voice_std.npy')

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)


### 3.2 Model Architecture

In [ ]:
class VoiceCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(VoiceCNN, self).__init__()

        # Input: [B, 40, 50]
        self.conv1 = nn.Conv1d(in_channels=40, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)  # 50 -> 25

        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.AdaptiveAvgPool1d(1)      # 25 -> 1

        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.squeeze(-1)  # [B, 32]
        x = self.fc(x)     # [B, C]
        return x

model = VoiceCNN(NUM_CLASSES).to(device)
print(f'✅ Model Initialized on {device}')
print(model)


### 3.3 Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f'🚀 Starting training for {EPOCHS} epochs...')

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / max(total, 1)
    avg_loss = running_loss / max(len(train_loader), 1)

    model.eval()
    test_correct = 0
    test_total = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            y_true.extend(labels.detach().cpu().numpy().tolist())
            y_pred.extend(predicted.detach().cpu().numpy().tolist())

    test_acc = 100 * test_correct / max(test_total, 1)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.1f}% | Test Acc: {test_acc:.1f}%')

print('✅ Training Complete.')

final_acc = accuracy_score(y_true, y_pred) * 100.0
print(f'Final Test Accuracy: {final_acc:.2f}%')
print(classification_report(y_true, y_pred, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Voice CNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()


## 4. Export to C++ Header (`voice_cnn_weights.h`)
This extracts trained parameters from `model.state_dict()` and formats them for HLS code.

In [ ]:
def export_pytorch_weights(model, filename='voice_cnn_weights.h'):
    print(f'Exporting weights to {filename}...')

    params = model.to('cpu').state_dict()

    with open(filename, 'w') as f:
        f.write('#ifndef VOICE_CNN_WEIGHTS_H\n#define VOICE_CNN_WEIGHTS_H\n\n')
        f.write('#include "voice_typedefs.h"\n\n')

        total_params = 0
        for name, tensor in params.items():
            clean_name = name.replace('.', '_').replace('weight', 'w').replace('bias', 'b')
            data = tensor.numpy().flatten()
            size = len(data)
            total_params += size

            print(f'Processing {name} -> {clean_name} ({size} elements)')

            f.write(f'// PyTorch Layer: {name} (Shape: {tuple(tensor.shape)})\n')
            f.write(f'static const data_t {clean_name}[{size}] = {{\n')

            for i, val in enumerate(data):
                f.write(f'{val:.6f}')
                if i < size - 1:
                    f.write(', ')
                if (i + 1) % 10 == 0:
                    f.write('\n    ')
            f.write('\n};\n\n')

        f.write('#endif // VOICE_CNN_WEIGHTS_H\n')

    print(f'\nDone! Total parameters: {total_params}')
    print("File saved as 'voice_cnn_weights.h'. Upload this to Vitis HLS.")


export_pytorch_weights(model, 'voice_cnn_weights.h')
